In [ ]:
!pip install optuna
!pip install flowio

!pip install git+https://github.com/enricodidoli-source/memory-save.git
!pip install git+https://github.com/enricodidoli-source/seed-manager.git

In [ ]:



import os

TF_DETERMINISTIC_OPS = '1'
TF_CUDNN_DETERMINISTIC = '1'
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'  # For CuDNN ops

import optuna
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import pickle
import time
import yaml
import tensorflow as tf
from tensorflow import keras

from memory_utils import clear_RAM, MemorySave
from seed_manager import SeedManager


In [ ]:

#fixed_path = '/content/drive/MyDrive/BinCellCNN/'
fixed_path = 'C:/Users/admin/BinCellCNN/'

#whole_exp = 'clinical_try_1'
whole_exp = 'clinical'

if fixed_path not in sys.path:
    sys.path.append(fixed_path)

modules_dir = f'{fixed_path}src/'
if modules_dir not in sys.path:
    sys.path.append(modules_dir)

experiment_path = f'{fixed_path}experiments/experiment_{whole_exp}/'
os.makedirs(experiment_path, exist_ok=True)

config_file_path = os.path.join(experiment_path, "test_clinical_config.yml")
with open(config_file_path, "r") as f:
    config = yaml.safe_load(f)

# --- Data ---
new_data_folder_path = config['new_data_folder_path']

extension = config['extension']
label_column = config['label_column']
remove_columns = config['remove_columns']  

# --- Seed parameters ---
starting_seed = config['starting_seed']


for key, value in config.items():
    print(f"{key}: {value}")




In [ ]:

decache_files = ['src.shared.utils', 'src.shared.timepoints_elaboration', 'src.BinCellCNN.model', 'src.clinical']

# Remove from cache
from src.shared.utils import remove_from_cache
remove_from_cache(decache_files)

from src.shared.timepoints_elaboration import load_data
from src.BinCellCNN.model import BinCellCNN
from src.clinical import classify_new_samples, load_new_data


# Single Cell Run

In [ ]:

new_unlabelled_samples, samples_info_df = load_new_data(config, new_data_folder_path)

seed_manager = SeedManager(starting_seed=starting_seed, LOPO_fold_idx=0)

test_samples_preds, best_thr = classify_new_samples(experiment_path = experiment_path,
                                                new_unlabelled_samples = new_unlabelled_samples,
                                                seed_manager = seed_manager,
                                                samples_info_df = samples_info_df)